# Hakhel reception history


## Table of contents (ToC)<a class="anchor" id="TOC"></a>

* <a href="#bullet1">1 - Purpose, method and limits</a>
* <a href="#bullet2">2 - Set up dependencies</a>
* <a href="#bullet3">3 - Retrieve data</a>
    * <a href="#bullet3x1">3.1 - Connect to Sefaria</a>
    * <a href="#bullet3x2">3.2 - Search for relevant passages</a>
    * <a href="#bullet3x3">3.3 - Follow links from Deuteronomy 31:12</a>
    * <a href="#bullet3x4">3.4 - Retrieve the full texts</a>
* <a href="#bullet4">4 - Curated chronological map</a>
    * <a href="#bullet4x1">4.1 - Compare periods and themes</a>
* <a href="#bullet5">5 - Reception-history synthesis</a>
    * <a href="#bullet5x1">5.1 - The rabbinic problem is not whether children are present</a>
    * <a href="#bullet5x2">5.2 - The saying in later interpretation</a>
    * <a href="#bullet5x3">5.3 - Medieval interpreters reopen the scriptural rationale</a>
    * <a href="#bullet5x4">5.4 - Later interpreters differentiate kinds of children</a>
    * <a href="#bullet5x5">5.5 - Synagogue reception produces extension and resistance</a>
    * <a href="#bullet5x6">5.6 - Hasidic readings of what children receive</a>
    * <a href="#bullet5x7">5.7 - Modern readings of covenant renewal</a>
* <a href="#bullet6">6 - Main conclusion</a>
* <a href="#bullet7">7 - Export reproducible results</a>
    * <a href="#bullet7x1">7.1 - Check the results</a>
* <a href="#bullet8">8 - Required libraries</a>
* <a href="#bullet9">9 - Responsible disclosure</a>
* <a href="#bullet10">10 - Licences</a>
* <a href="#bullet11">11 - Notebook details</a>


# 1 - Purpose, method and limits<a class="anchor" id="bullet1"></a>
##### [Back to ToC](#TOC)

This notebook traces how Jewish interpreters received the inclusion of `הטף` (“the little ones”) in Deut 31:12. Of particular interest is the rabbinic question and answer, `טף למה באין? ... ליתן שכר למביאיהן`: why do the little ones come, and why is merit assigned to those who bring them?

The source data and the interpretive layers remain separate. Sefaria supplies search hits, links, and text. `evidence.csv` is a manually curated chronological reading table. The final discovery tags are a researcher-supplied keyword index for navigating the retrieved texts; Sefaria does not supply them.

The retrieval plan specifies the search terms and reference seeds, including Maor VaShemesh, Vayeilech 5. The notebook retrieves the texts through Sefaria and saves the results with their run metadata.

In this notebook we will use three Sefaria MCP operations:

- keyword search to discover reception witnesses;
- link retrieval from Deut 31:12 to identify direct commentary;
- full-text retrieval for close reading.

Run the cells in order to save a dated run under `results/source_runs/`. Each run contains raw responses, reading tables, and a validation audit. Search indexes and available text versions can change, so the audit forms part of the evidence.

The resulting chronology is a reception map, not a claim that Sefaria contains every relevant source. Search results should be checked against critical editions when textual variants or dating matter.

# 2 - Set up dependencies<a class="anchor" id="bullet2"></a>
##### [Back to ToC](#TOC)

The notebook needs a few Python libraries to contact Sefaria and work with the returned data. The next cell shows how to install them from the requirements file. Leave the installation line commented out when the libraries are already available.


In [1]:
# Dependencies are listed in reception_history/requirements.txt.
# Uncomment the next line only when preparing a new environment.
# %pip install -q -r "requirements.txt"


# 3 - Retrieve data<a class="anchor" id="bullet3"></a>
##### [Back to ToC](#TOC)

We first set the location of the notebook files and the Sefaria connection. The retrieval plan contains the search terms and the references to fetch. A folder named with the date and time keeps each run’s results together.


In [2]:
# Method, paths, and one fresh retrieval run
from __future__ import annotations

from datetime import datetime, timezone
from zoneinfo import ZoneInfo
import uuid
import sys
import importlib.metadata as package_metadata
import json
import re
from pathlib import Path
from typing import Any

import pandas as pd
from tqdm.auto import tqdm
from mcp import ClientSession
from mcp.client.sse import sse_client

MCP_SERVER_URL = "https://mcp.sefaria.org/sse"

HERE = Path.cwd()
if not (HERE / "results").exists() and (HERE / "reception_history").exists():
    HERE = HERE / "reception_history"
RESULTS = HERE / "results"
RESULTS.mkdir(exist_ok=True)

strategy = json.loads((HERE / "retrieval_plan.json").read_text(encoding="utf-8"))
RUN_ID = datetime.now(ZoneInfo("Europe/Amsterdam")).strftime("live_%Y-%m-%d_%H%M%S") + "_" + uuid.uuid4().hex[:8]
RUN_DIR = RESULTS / "source_runs" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
STARTED_AT = datetime.now(ZoneInfo("Europe/Amsterdam")).isoformat()
SEARCH_QUERIES = strategy["queries"]
SEED_REFS = strategy["seed_references"]

print("Fresh retrieval:", RUN_ID)
print("Search queries:", len(SEARCH_QUERIES))

D:\Onedrive\GitHub\Hakhel\.tmp\mcp130-verified\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fresh retrieval: live_2026-10-26_105712_58edc937
Search queries: 12


## 3.1 - Connect to Sefaria<a class="anchor" id="bullet3x1"></a>
##### [Back to ToC](#TOC)

The next cell defines the small helper functions used to contact Sefaria. They send a request, read the response, and turn it into data that Python can use. If Sefaria reports an error, the notebook shows it instead of treating it as a successful result.


In [3]:
# Handle MCP responses
def decode_result(result: Any) -> Any:
    content = getattr(result, "content", result)
    if isinstance(content, list) and content:
        content = getattr(content[0], "text", content[0])
    if isinstance(content, str):
        try:
            payload = json.loads(content)
        except json.JSONDecodeError:
            return content
        if isinstance(payload, dict) and "result" in payload:
            inner = payload["result"]
            if isinstance(inner, str):
                try:
                    return json.loads(inner)
                except json.JSONDecodeError:
                    return inner
            return inner
        return payload
    return content


async def call_tool(name: str, arguments: dict[str, Any]) -> Any:
    async with sse_client(MCP_SERVER_URL) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool(name, arguments)
            if getattr(result, "isError", False):
                raise RuntimeError(f"Sefaria {name}: {decode_result(result)}")
            return decode_result(result)


async def list_tools() -> list[dict[str, Any]]:
    async with sse_client(MCP_SERVER_URL) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            response = await session.list_tools()
            return [tool.model_dump() for tool in response.tools]

## 3.2 - Search for relevant passages<a class="anchor" id="bullet3x2"></a>
##### [Back to ToC](#TOC)

We now run each search term in the retrieval plan. Each result keeps the term that found it, so we can trace how a passage entered the study. The raw responses are saved before repeated references are removed from the reading table. The table below shows the first results.


In [4]:
# Search for reception witnesses
def result_rows(payload: Any, query: str) -> list[dict[str, Any]]:
    """Normalize result objects while retaining their search query."""
    if isinstance(payload, dict):
        payload = payload.get("results", payload.get("hits", []))
    rows = []
    for item in payload or []:
        row = dict(item)
        row["search_query"] = query
        rows.append(row)
    return rows


raw_searches = []
rows = []
for query in tqdm(SEARCH_QUERIES, desc="Sefaria searches"):
    payload = await call_tool("text_search", {"query": query, "size": strategy["search_size"]})
    raw_searches.append({"query": query, "payload": payload})
    rows.extend(result_rows(payload, query))
    with (RUN_DIR / "raw_search_payloads.jsonl").open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(raw_searches[-1], ensure_ascii=False) + "\n")

search_df = pd.DataFrame(rows)
if not search_df.empty and "ref" in search_df:
    search_df = search_df.drop_duplicates(subset=["ref"]).reset_index(drop=True)

search_df.head(20)

Sefaria searches:   0%|          | 0/12 [00:00<?, ?it/s]

Sefaria searches:   8%|▊         | 1/12 [00:04<00:51,  4.65s/it]

Sefaria searches:  17%|█▋        | 2/12 [00:08<00:41,  4.14s/it]

Sefaria searches:  25%|██▌       | 3/12 [00:10<00:28,  3.13s/it]

Sefaria searches:  33%|███▎      | 4/12 [00:12<00:23,  2.89s/it]

...

Sefaria searches:  83%|████████▎ | 10/12 [00:28<00:05,  2.75s/it]

Sefaria searches:  92%|█████████▏| 11/12 [00:33<00:03,  3.35s/it]

Sefaria searches: 100%|██████████| 12/12 [00:35<00:00,  3.01s/it]

Sefaria searches: 100%|██████████| 12/12 [00:35<00:00,  2.95s/it]

,ref,categories,text_snippet,search_query
0,Deuteronomy 31:12,"[Tanakh, Torah]",<b>הַקְהֵל</b> <b>אֶת</b>־<b>הָעָם</b> <b>הָאֲ...,הקהל את העם האנשים והנשים והטף
1,Chagigah 3a:17,"[Talmud, Bavli, Seder Moed]",<b>הקהל</b> <b>את</b> <b>העם</b> <b>האנשים</b>...,הקהל את העם האנשים והנשים והטף
2,"Mishneh Torah, Festival Offering 3:1","[Halakhah, Mishneh Torah, Sefer Korbanot]",מִצְוַת עֲשֵׂה לְהַקְהִיל כָּל יִשְׂרָאֵל <b>א...,הקהל את העם האנשים והנשים והטף
3,Chagigah 3a:2,"[Talmud, Bavli, Seder Moed]",לְעִנְיַן רְאִיָּה גָּמַר ״רְאִיָּה״ ״רְאִיָּה...,הקהל את העם האנשים והנשים והטף
4,Sefer HaChinukh 612:1,"[Halakhah, Sifrei Mitzvot]",לְהַקְהִיל כָּל יִשְׂרָאֵל בְּחַג הַסֻּכּוֹת –...,הקהל את העם האנשים והנשים והטף
5,Jerusalem Talmud Sotah 3:4:6,"[Talmud, Yerushalmi, Seder Nashim]",וְהִקְבִילוּ <b>אֶת</b> רִבִּי יְהוֹשֻׁעַ בִּב...,הקהל את העם האנשים והנשים והטף
6,Sefer Chasidim 313:1,"[Halakhah, Rishonim]",כאלו מלמדה תיפלות זהו עומק תלמוד וטעמי המצות ו...,הקהל את העם האנשים והנשים והטף
7,Bamidbar Rabbah 14:4,"[Midrash, Aggadah, Midrash Rabbah]",עַל פִּי כֵן אִי אֶפְשָׁר לְבֵית הַמִּדְרָשׁ ב...,הקהל את העם האנשים והנשים והטף
8,Avot DeRabbi Natan 18:2,"[Talmud, Bavli, Minor Tractates]",אמר להם ובמה היתה ההגדה היום אמרו בפרשת <b>הקה...,הקהל את העם האנשים והנשים והטף
9,"Abudarham, Pesach, Festival Prayers 12","[Halakhah, Rishonim]",ועוד טעם אחר לפי שאבותינו לא קבלו <b>את</b> הת...,הקהל את העם האנשים והנשים והטף


## 3.3 - Follow links from Deuteronomy 31:12<a class="anchor" id="bullet3x3"></a>
##### [Back to ToC](#TOC)

Keyword searches can miss a commentary that does not repeat our search terms. We therefore also ask Sefaria for the texts linked directly to Deut 31:12. The table shows the reference, the kind of link, and the category assigned by Sefaria.


In [5]:
# Retrieve links from the anchor verse
linked = await call_tool(
    "get_links_between_texts",
    {"reference": "Deuteronomy 31:12", "with_text": "1"},
)
links_df = pd.DataFrame(linked or [])

links_df[["ref", "type", "category"]].head(30)

,ref,type,category
0,Ramban on Deuteronomy 31:12:1,commentary,Commentary
1,Rashi on Deuteronomy 31:12:1,commentary,Commentary
2,Rashi on Deuteronomy 31:12:2,commentary,Commentary
3,Ibn Ezra on Deuteronomy 31:12:1,commentary,Commentary
4,Ibn Ezra on Deuteronomy 31:12:2,commentary,Commentary
5,Onkelos Deuteronomy 31:12,targum,Targum
6,Targum Jonathan on Deuteronomy 31:12,targum,Targum
7,Kli Yakar on Deuteronomy 31:12:1,commentary,Commentary
8,Kli Yakar on Deuteronomy 31:12:2,commentary,Commentary
9,Kli Yakar on Deuteronomy 31:12:3,commentary,Commentary


## 3.4 - Retrieve the full texts<a class="anchor" id="bullet3x4"></a>
##### [Back to ToC](#TOC)

The next cell brings together the planned references, the search results, and the links from the verse. It retrieves each reference once and keeps the available Hebrew and English texts. Simple web formatting is removed for easier reading. Raw responses are saved separately; failed or empty responses are recorded for later review.


In [6]:
# Retrieve full texts for close reading
def strip_html(value: Any) -> str:
    if isinstance(value, list):
        return " ".join(strip_html(v) for v in value)
    return re.sub(r"<[^>]+>", "", str(value or "")).strip()


def normalize_text(ref: str, payload: Any) -> dict[str, Any]:
    """Keep the first available Hebrew and English versions."""
    versions = payload.get("versions", []) if isinstance(payload, dict) else []
    row = {"ref": payload.get("ref", ref) if isinstance(payload, dict) else ref}
    for version in versions:
        language = version.get("languageFamilyName", "").lower()
        if language == "hebrew" and "hebrew" not in row:
            row["hebrew"] = strip_html(version.get("text"))
        if language == "english" and "english" not in row:
            row["english"] = strip_html(version.get("text"))
    return row


discovered = [] if search_df.empty else search_df["ref"].dropna().tolist()
linked_refs = [] if links_df.empty else links_df["ref"].dropna().tolist()
refs = list(dict.fromkeys(SEED_REFS + discovered + linked_refs))

texts = []
for ref in tqdm(refs, desc="Full texts"):
    try:
        payload = await call_tool(
            "get_text", {"reference": ref, "version_language": "both"}
        )
        row = normalize_text(ref, payload)
        if not row.get("hebrew") and not row.get("english"):
            raise ValueError("Response contains no Hebrew or English text")
        texts.append(row)
        with (RUN_DIR / "raw_full_text_payloads.jsonl").open("a", encoding="utf-8") as handle:
            handle.write(json.dumps({"requested_ref": ref, "payload": payload}, ensure_ascii=False) + "\n")
    except Exception as exc:
        texts.append({"ref": ref, "error": repr(exc)})
texts_df = pd.DataFrame(texts)

texts_df.head(20)

Full texts:   0%|          | 0/345 [00:00<?, ?it/s]

Full texts:   0%|          | 1/345 [00:01<10:27,  1.83s/it]

Full texts:   1%|          | 2/345 [00:03<08:16,  1.45s/it]

Full texts:   1%|          | 3/345 [00:04<07:14,  1.27s/it]

Full texts:   1%|          | 4/345 [00:05<07:05,  1.25s/it]

...

Full texts:  99%|█████████▉| 343/345 [07:15<00:02,  1.23s/it]

Full texts: 100%|█████████▉| 344/345 [07:17<00:01,  1.40s/it]

Full texts: 100%|██████████| 345/345 [07:19<00:00,  1.60s/it]

Full texts: 100%|██████████| 345/345 [07:19<00:00,  1.27s/it]

,ref,english,hebrew,error
0,Deuteronomy 31:10-13,And Moses instructed them as follows: Every se...,וַיְצַ֥ו מֹשֶׁ֖ה אוֹתָ֣ם לֵאמֹ֑ר מִקֵּ֣ץ&thins...,NaN
1,Tosefta Sotah 7:6,It so happened with Rabbi Yochanan ben Berokah...,מעשה בר' יוחנן בן ברוקה ור' אלעזר &lt;בן&gt; ח...,NaN
2,Jerusalem Talmud Chagigah 1:1:3,"7The same text is in Soṭah3:4 (Notes 107–109, ...",דְּתַנֵּי. מַעֲשֶׂה בְרִבִּי יוֹחָנַן בֶּן בְּ...,NaN
3,Chagigah 3a:17,They said to him that Rabbi Elazar ben Azarya ...,״הַקְהֵל אֶת הָעָם הָאֲנָשִׁים וְהַנָּשִׁים וְ...,NaN
4,Mishnah Sotah 7:8,How is the portion of the Torah that is read b...,פָּרָשַׁת הַמֶּלֶךְ כֵּיצַד. מוֹצָאֵי יוֹם טוֹ...,NaN
5,"Mishneh Torah, Festival Offering 3",It is a positive commandment to gather togethe...,מִצְוַת עֲשֵׂה לְהַקְהִיל כָּל יִשְׂרָאֵל אֲנָ...,NaN
6,Rashi on Deuteronomy 31:12,"האנשים [ASSEMBLE THE PEOPLE,] MEN [WOMEN, AND ...",האנשים. לִלְמֹד: והנשים. לִשְׁמֹעַ: והטף. לָמּ...,NaN
7,"Abarbanel on Torah, Deuteronomy 31:9:5",NaN,וצוה הקהל את העם האנשים והנשים והטף. והנראה כד...,NaN
8,Kli Yakar on Deuteronomy 31:12,"Gather the people, the men, and the women, and...",הַקְהֵל אֶת הָעָם הָאֲנָשִׁים וְהַנָּשִׁים וְה...,NaN
9,"Sefat Emet, Deuteronomy, Vayeilech 3:2",The Gemara (Chagigah 3a) explains that childre...,"טף למה באין לתת שכר למביאיהן. וקשה ממ""נ אי אין...",NaN


# 4 - Curated chronological map<a class="anchor" id="bullet4"></a>
##### [Back to ToC](#TOC)

The retrieved texts provide the source material. The table below is the close-reading layer: selected witnesses are ordered by period and grouped by their main interpretive theme. It reads the existing `evidence.csv` file. The categories, chronology, and descriptions are AI assisted and researcher-reviewed, and remain open to revision.


In [7]:
# Curated chronological map
evidence = pd.read_csv(RESULTS / "evidence.csv")
evidence

,period,date_range,reference,genre,reception_move,role_of_the_little_ones,theme
0,Biblical base,textual starting point,Deuteronomy 31:10-13,Torah,Commands a septennial public Torah reading bef...,They are explicitly present and 31:13 associat...,covenant inclusion
1,Early rabbinic,2nd-3rd c.,Tosefta Sotah 7:6,Tosefta,Frames the inclusion as a deliberate exegetica...,Their attendance produces reward for those who...,reward to bringers
2,Early rabbinic,3rd-6th c.,Jerusalem Talmud Chagigah 1:1:3,Yerushalmi,Transmits the Elazar ben Azariah interpretatio...,Their presence merits the adults who bring them.,reward to bringers
3,Early rabbinic,3rd-6th c.,Jerusalem Talmud Chagigah 1:1:2,Yerushalmi,Uses Hakhel to argue that a minor can be oblig...,The category טף becomes a legal analogy for mi...,minor obligation
4,Early rabbinic,3rd-6th c.,Chagigah 3a:17,Bavli,Canonizes the question and answer and calls th...,They come so reward may be given to those who ...,reward to bringers
5,Early rabbinic,3rd-7th c.,Avot DeRabbi Natan 18:2,Minor tractate,Retells the study-house narrative and preserve...,Their attendance rewards their bringers.,reward to bringers
6,Early rabbinic,late antique/early medieval,Tractate Soferim 18:6,Minor tractate,Extends the Hakhel saying into synagogue custo...,Attendance habituates children and rewards bot...,synagogue formation
7,Targumic,late antique/medieval,Targum Jonathan on Deuteronomy 31:12,Targum,Builds the rabbinic differentiation directly i...,Children share in the benefit or reward of tho...,translation as interpretation
8,Medieval,11th c.,Rashi on Deuteronomy 31:12,Biblical commentary,Makes the Bavli's three-part division the defa...,They are present to bring merit to those who c...,reward to bringers
9,Medieval,12th c.,Ibn Ezra on Deuteronomy 31:12:3,Biblical commentary,Reads hearing as the trigger for questioning a...,Young children can learn through hearing and a...,child learning


## 4.1 - Compare periods and themes<a class="anchor" id="bullet4x1"></a>
##### [Back to ToC](#TOC)

This table counts the selected witnesses for each combination of period and theme. It helps us see how the curated examples are distributed. These counts describe our selection; they do not measure how common a reading was in Jewish interpretation as a whole.


In [8]:
# Summarize the curated evidence by period and theme
pd.crosstab(evidence["period"], evidence["theme"])

theme,Sinai reenactment,age differentiation,age threshold,child learning,communal solidarity,counter-reception,covenant inclusion,covenant renewal,formation,inclusive hearing,minor obligation,reward to bringers,spiritual imprint,synagogue formation,translation as interpretation
period,,,,,,,,,,,,,,,
Biblical base,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
Early modern,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0
Early rabbinic,0,0,0,0,0,0,0,0,0,0,1,4,0,1,0
Hasidic,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0
Late medieval,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
Medieval,1,0,0,1,0,0,0,0,1,1,0,1,0,0,0
Modern,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0
Modern halakhic,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0
Targumic,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1


# 5 - Reception-history synthesis<a class="anchor" id="bullet5"></a>
##### [Back to ToC](#TOC)

The links in this section point to Sefaria locations represented in the evidence package. They open the text fragment used for the corresponding observation, rather than a general search page. Where the stored evidence is a passage range, the link may narrow that range to the particular verse under discussion.

## 5.1 - The rabbinic problem is not whether children are present<a class="anchor" id="bullet5x1"></a>
##### [Back to ToC](#TOC)

[Deuteronomy 31:10–13](https://www.sefaria.org/Deuteronomy.31.10-13) already commands their presence. The [early rabbinic
intervention](https://www.sefaria.org/Chagigah.3a.17) asks what *purpose* that presence serves when infants or very young children cannot perform the stated acts of learning. The answer places the reward on the adults who bring them.

## 5.2 - The saying in later interpretation<a class="anchor" id="bullet5x2"></a>
##### [Back to ToC](#TOC)

The [Bavli's three-part formula](https://www.sefaria.org/Chagigah.3a.17) also appears in [Targum Jonathan](https://www.sefaria.org/Targum_Jonathan_on_Deuteronomy.31.12), [Rashi](https://www.sefaria.org/Rashi_on_Deuteronomy.31.12), the [Mishneh Torah](https://www.sefaria.org/Mishneh_Torah%2C_Festival_Offering.3), the [Sefer HaChinukh](https://www.sefaria.org/Sefer_HaChinukh.612.1), and later commentary. It then supports arguments about women's Torah hearing, minors' ritual participation, parental duty, and the social meaning of public worship.

## 5.3 - Medieval interpreters reopen the scriptural rationale<a class="anchor" id="bullet5x3"></a>
##### [Back to ToC](#TOC)

[Ibn Ezra](https://www.sefaria.org/Ibn_Ezra_on_Deuteronomy.31.12.3) and especially [Abarbanel](https://www.sefaria.org/Abarbanel_on_Torah%2C_Deuteronomy.31.9.5) resist reducing children to passive occasions for adult reward. [Deuteronomy 31:13](https://www.sefaria.org/Deuteronomy.31.13) says that children who do not yet know will hear and learn. This raises a question: how does reward for the adults relate to the children's own hearing and learning?

## 5.4 - Later interpreters differentiate kinds of children<a class="anchor" id="bullet5x4"></a>
##### [Back to ToC](#TOC)

[Kli Yakar](https://www.sefaria.org/Kli_Yakar_on_Deuteronomy.31.12) and the [Minchat Chinukh](https://www.sefaria.org/Minchat_Chinukh.612.1.1) divide `טף` by developmental stage:
children capable of education can benefit directly, while younger children raise questions about the adult obligation to bring, the lower age boundary, and the meaning of reward.

## 5.5 - Synagogue reception produces extension and resistance<a class="anchor" id="bullet5x5"></a>
##### [Back to ToC](#TOC)

[Tractate Soferim](https://www.sefaria.org/Tractate_Soferim.18.6) turns Hakhel into precedent for bringing young girls to
synagogue. The [*Shenei Luchot HaBerit*](https://www.sefaria.org/Shenei_Luchot_HaBerit%2C_Aseret_HaDibrot%2C_Tamid%2C_Ner_Mitzva.130) preserves a counter-reading: if children disrupt prayer and desecrate the space, the bringer should expect punishment rather than reward.

## 5.6 - Hasidic readings of what children receive<a class="anchor" id="bullet5x6"></a>
##### [Back to ToC](#TOC)

[Sefat Emet](https://www.sefaria.org/Sefat_Emet%2C_Deuteronomy%2C_Vayeilech.3.2) argues that children can receive an inward illumination even without conceptual understanding. [Maor VaShemesh](https://www.sefaria.org/Maor_VaShemesh%2C_Vayeilech.5) places them within the love and unity of Israel that makes Torah hearing possible.

## 5.7 - Modern readings of covenant renewal<a class="anchor" id="bullet5x7"></a>
##### [Back to ToC](#TOC)

[A modern reading retained in the evidence package](https://www.sefaria.org/Covenant_and_Conversation%3B_Deuteronomy%3B_Renewal_of_the_Sinai_Covenant%2C_Vayeilech%2C_To_Renew_Our_Days.6) describes Hakhel as national covenant renewal and inclusive public education. The little ones signify that Torah belongs to the next generation before they can fully understand or consent to it.

# 6 - Main conclusion<a class="anchor" id="bullet6"></a>
##### [Back to ToC](#TOC)

The reception history supplies four recurring answers to why the little
ones come:

1. **adult merit** — bringing them is itself rewarded;
2. **formation** — repeated presence habituates future participants;
3. **direct reception** — children hear, learn, or receive a spiritual
   impression according to their capacity;
4. **corporate identity** — their presence makes the assembly visibly
   intergenerational and covenantal.

These answers coexist. The readings differ in how much they emphasise the adults who bring the children and the children who hear the Torah.

# 7 - Export reproducible results<a class="anchor" id="bullet7"></a>
##### [Back to ToC](#TOC)

We now save the search table, the links, and the full texts in this run’s folder. We also save the retrieval plan and the list of tools available from Sefaria. These files show what was requested and which service supplied the data.


In [9]:
# Export the retrieval tables and configuration for this run.
search_df.to_csv(RUN_DIR / "search_results.csv", index=False, encoding="utf-8-sig")
links_df.to_csv(RUN_DIR / "anchor_links.csv", index=False, encoding="utf-8-sig")
texts_df.to_csv(RUN_DIR / "full_texts.csv", index=False, encoding="utf-8-sig")
tools = await list_tools()
(RUN_DIR / "mcp_tool_inventory.json").write_text(json.dumps(tools, ensure_ascii=False, indent=2), encoding="utf-8")
(RUN_DIR / "retrieval_plan.json").write_text(json.dumps(strategy, ensure_ascii=False, indent=2), encoding="utf-8")
print("Run exported to", RUN_DIR)

Run exported to C:\Users\tonyj\AppData\Local\Temp\hakhel-mcp130-4wp8lnvi\Hakhel\reception_history\results\source_runs\live_2026-10-26_105712_58edc937


## 7.1 - Check the results<a class="anchor" id="bullet7x1"></a>
##### [Back to ToC](#TOC)

The final code cell makes the reading tables and checks the run. It keeps every search-term and reference relationship, while giving each full-text reference one row in the reading table. The discovery tags are simple keyword labels for finding passages; they are separate from the curated interpretation.

The audit lists missing curated references, failed text requests, and searches that reached the result limit. A search at the limit may have more matches than the tool returned. Check these entries before relying on the results.


In [10]:
# Build and validate the reading tables from this run alone.
def discovery_tags(row: pd.Series) -> str:
    """Assign navigation tags using the explicit keyword rules below."""
    haystack = " ".join(
        str(row.get(column, ""))
        for column in ["ref", "hebrew", "english", "raw_text"]
    )
    tags = []
    if any(term in haystack for term in ["Deuteronomy 31", "הקהל את העם", "האנשים והנשים והטף"]):
        tags.append("Torah instruction")
    if any(term in haystack for term in ["טף למה", "ליתן שכר", "bring them", "bringing them"]):
        tags.append("rabbinic reason for children")
    if any(term in haystack for term in ["חגיגה", "Chagigah"]):
        tags.append("Chagigah discussion")
    if any(term in haystack for term in ["קידושין", "Kiddushin", "time-bound"]):
        tags.append("mitzvah obligation discussion")
    return "; ".join(tags) or "related reference"



search_source_rows = pd.DataFrame([
    dict(row, source_run=RUN_ID)
    for item in raw_searches
    for row in result_rows(item["payload"], item["query"])
])
search_source_rows.to_csv(RUN_DIR / "search_sources.csv", index=False, encoding="utf-8-sig")
consolidated_search = search_source_rows.groupby("ref", as_index=False, sort=True).agg(
    source_runs=("source_run", "first"),
    search_queries=("search_query", lambda values: " | ".join(dict.fromkeys(values))),
)
consolidated_search.to_csv(RUN_DIR / "search_refs.csv", index=False, encoding="utf-8-sig")
full_text_source_rows = texts_df.assign(source_run=RUN_ID)
full_text_source_rows.to_csv(RUN_DIR / "text_sources.csv", index=False, encoding="utf-8-sig")
consolidated_texts = texts_df.drop_duplicates("ref").copy()
consolidated_texts["discovery_tags"] = consolidated_texts.apply(discovery_tags, axis=1)
consolidated_texts.to_csv(RUN_DIR / "texts.csv", index=False, encoding="utf-8-sig")

successful = consolidated_texts.fillna("")
successful = successful.loc[successful.get("hebrew", pd.Series("", index=successful.index)).ne("") | successful.get("english", pd.Series("", index=successful.index)).ne("")]
new_refs = set(successful["ref"])
audit = {
    "run_id": RUN_ID, "started_at": STARTED_AT,
    "completed_at": datetime.now(ZoneInfo("Europe/Amsterdam")).isoformat(),
    "started_at_utc": datetime.fromisoformat(STARTED_AT).astimezone(timezone.utc).isoformat(),
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "software_versions": {
        "python": sys.version.split()[0],
        **{name: package_metadata.version(name) for name in ("mcp", "pandas", "nbformat")},
    },
    "endpoint": MCP_SERVER_URL, "queries": len(SEARCH_QUERIES),
    "requested_texts": len(refs), "successful_texts": len(successful),
    "missing_curated_references": sorted(set(evidence["reference"]) - new_refs),
    "queries_at_result_limit": [item["query"] for item in raw_searches if len(result_rows(item["payload"], item["query"])) >= strategy["search_size"]],
    "retrieval_errors": json.loads(
        texts_df.loc[
            texts_df.get("error", pd.Series(index=texts_df.index, dtype=str)).notna()
        ].to_json(orient="records", force_ascii=False)
    ),
}
(RUN_DIR / "audit.json").write_text(json.dumps(audit, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
display(pd.Series(audit))
print("Run exported. Review missing curated references, retrieval errors, and search limits in audit.json.")


run_id                                          live_2026-10-26_105712_58edc937
started_at                                     2026-10-26T10:57:12.506613+01:00
completed_at                                   2026-10-26T11:05:13.069540+01:00
started_at_utc                                 2026-10-26T09:57:12.506613+00:00
completed_at_utc                               2026-10-26T10:05:13.069562+00:00
software_versions             {'python': '3.14.2', 'mcp': '1.30.0', 'pandas'...
endpoint                                            https://mcp.sefaria.org/sse
queries                                                                      12
requested_texts                                                             345
successful_texts                                                            344
missing_curated_references                                                   []
queries_at_result_limit       [הקהל את העם האנשים והנשים והטף, ליתן שכר למבי...
retrieval_errors              [{'ref': '

Run exported. Review missing curated references, retrieval errors, and search limits in audit.json.


# 8 - Required libraries<a class="anchor" id="bullet8"></a>
##### [Back to ToC](#TOC)

Install the dependencies from [requirements.txt](requirements.txt) in the environment used by this notebook:

```powershell
python -m pip install -r requirements.txt
```

Run this command from the notebook directory. The [reproduction guide](../REPRODUCING.md) describes environment creation and kernel selection.


# 9 - Responsible disclosure<a class="anchor" id="bullet9"></a>
##### [Back to ToC](#TOC)

The author used OpenAI Codex to assist with Python code and data analysis. The author reviewed and edited the material and takes responsibility for the content and interpretation.

# 10 - Licences<a class="anchor" id="bullet10"></a>
##### [Back to ToC](#TOC)

The original code in this notebook is licensed under the [MIT License](../LICENSE).
The original research writing and explanatory text are licensed under [Creative Commons Attribution 4.0 International (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/).
Credit Tony Jurg, link the licence and repository, and indicate any changes when reusing the original text; see the [text licence](../LICENSE-TEXT.md).

Quoted source material, retrieved passages, and datasets retain their own terms.
See [Sources, attribution, and reuse](../DATA_RIGHTS.md) for their licences and attribution requirements.


# 11 - Notebook details<a class="anchor" id="bullet11"></a>
##### [Back to ToC](#TOC)

<div style="float: left;">
  <table>
    <tr><td><strong>Author</strong></td><td>Tony Jurg</td></tr>
    <tr><td><strong>Version</strong></td><td>1.7</td></tr>
    <tr><td><strong>Date</strong></td><td>October 26, 2026</td></tr>
  </table>
</div>
